In [0]:
%sql

SELECT
    COUNT(*)
FROM bootcamp.gold.dim_zona;

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.gold.dim_provincia_sf;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_provincia_sf
(
    provincia_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    provincia STRING NOT NULL,
    pais STRING NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP() COMMENT 'DateTime in UTC'
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension SNOWFLAKE Provincia";

INSERT INTO bootcamp.gold.dim_provincia_sf
(
    provincia,
    pais
)
SELECT
    DISTINCT provincia,
    pais
FROM bootcamp.gold.dim_zona;



In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_ciudad_sf;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_ciudad_sf
(
    ciudad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    ciudad STRING NOT NULL,
    provincia_id BIGINT NOT NULL COMMENT 'FK to dim_provincia_sf',    
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP() COMMENT 'DateTime in UTC'
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT 'Dimension SNOWFLAKE Ciudad';

INSERT INTO bootcamp.gold.dim_ciudad_sf
(
    ciudad,
    provincia_id
)
SELECT
    DISTINCT z.ciudad,
    p.provincia_id
FROM bootcamp.gold.dim_zona as z
INNER JOIN bootcamp.gold.dim_provincia_sf as p ON z.provincia = p.provincia AND z.pais = p.pais;


In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_zona_sf;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona_sf
(
    zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT 'PK',
    partido STRING NOT NULL,
    region STRING NOT NULL,
    ciudad_id BIGINT NOT NULL COMMENT 'FK to dim_ciudad_sf'
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT 'Dimension SNOWFLAKE zona';


INSERT OVERWRITE bootcamp.gold.dim_zona_sf
(
    partido,
    region,
    ciudad_id
)
SELECT
    DISTINCT z.partido,
    z.region,
    c.ciudad_id
FROM bootcamp.gold.dim_zona as z
INNER JOIN bootcamp.gold.dim_ciudad_sf as c ON z.ciudad = c.ciudad
ORDER BY partido;


In [0]:
%sql

SELECT * FROM bootcamp.gold.dim_zona;

In [0]:
%sql

SELECT
    z.zona_id,
    z.partido,
    z.region,
    c.ciudad,
    p.provincia,
    p.pais
FROM bootcamp.gold.dim_zona_sf z
INNER JOIN bootcamp.gold.dim_ciudad_sf c ON z.ciudad_id=c.ciudad_id
INNER JOIN bootcamp.gold.dim_provincia_sf p ON c.provincia_id=p.provincia_id;
